<a href="https://colab.research.google.com/github/Mohamed-Mohamed-Ibrahim/Code-Generation-and-Guarding/blob/SFT/sft/peft-sft-2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!apt install tree

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  tree
0 upgraded, 1 newly installed, 0 to remove and 98 not upgraded.
Need to get 47.9 kB of archives.
After this operation, 116 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tree amd64 2.0.2-1 [47.9 kB]
Fetched 47.9 kB in 0s (421 kB/s)

78Selecting previously unselected package tree.
(Reading database ... 129073 files and directories currently installed.)
Preparing to unpack .../tree_2.0.2-1_amd64.deb ...
7Progress: [  0%] [..........................................................] 87Progress: [ 20%] [###########...............................................] 8Unpacking tree (2.0.2-1) ...
7Progress: [ 40%] [#######################...................................] 8Setting up tree (2.0.2-1) ...
7Progress: [ 60%] [##################################........................] 8

In [2]:
!pip install peft transformers[torch] trl bitsandbytes datasets -U --q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 556.4/556.4 kB 13.2 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 518.9/518.9 kB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 34.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.3/512.3 kB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 111.6 MB/s eta 0:00:0000:010:01


In [3]:
import os
import random
import torch
from datasets import Dataset, load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, prepare_model_for_kbit_training

2025-12-25 04:47:16.014138: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766638036.215064      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766638036.272215      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1766638036.747843      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766638036.747878      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766638036.747882      55 computation_placer.cc:177] computation placer alr

In [4]:
dataset = load_dataset("flytech/python-codes-25k",split="train")

README.md: 0.00B [00:00, ?B/s]

python-codes-25k.json:   0%|          | 0.00/26.4M [00:00<?, ?B/s]

python-codes-25k.jsonl:   0%|          | 0.00/25.4M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/49626 [00:00<?, ? examples/s]

In [5]:
dataset

Dataset({
    features: ['output', 'instruction', 'input', 'text'],
    num_rows: 49626
})

In [6]:
shuffled_dataset = dataset.shuffle(seed=42)
train_dataset = shuffled_dataset.select(range(4000))
eval_dataset = shuffled_dataset.select(range(200))
test_dataset = shuffled_dataset.select(range(200))

In [7]:
print("="*100)
print(shuffled_dataset)
print("="*100)
print(shuffled_dataset[0]['input'])
print("="*100)
print(shuffled_dataset[0]['instruction'])
print("="*100)
print(shuffled_dataset[0]['text'])
print("="*100)
print(shuffled_dataset[0]['output'])
print("="*100)


Dataset({
    features: ['output', 'instruction', 'input', 'text'],
    num_rows: 49626
})

Write a Python program to generate a sorted list of unique numbers from 0 to 9 in random order
Write a Python program to generate a sorted list of unique numbers from 0 to 9 in random order Let's roll! The ball's in our court! ```python
import random

# generating a list of unique numbers from 0 to 9 in random order
random_numbers = random.sample(range(0, 10), 10)

# sort list of numbers 
random_numbers.sort()

# print sorted list of random numbers
print(random_numbers)
# Output: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
```
```python
import random

# generating a list of unique numbers from 0 to 9 in random order
random_numbers = random.sample(range(0, 10), 10)

# sort list of numbers 
random_numbers.sort()

# print sorted list of random numbers
print(random_numbers)
# Output: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
```


In [8]:
# HF hub ID
BASE_MODEL_NAME = "Qwen/Qwen2-1.5B-Instruct"
BASE_MODEL_NAME = "arnir0/Tiny-LLM"

# Push adapter artifacts post training
OUTPUT_DIR = "./qlora-peft-output"
ADAPTER_DIR = os.path.join(OUTPUT_DIR, "adapter")

## Hyperparameters

In [9]:
lora_r = 16
lora_target_modules = "all-linear"
bs =  1
gradient_accumulation_steps = 4
epochs = 1
lr = 2e-4
optim = "paged_adamw_8bit"
max_length = 1024

In [10]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

tokenizer_config.json:   0%|          | 0.00/930 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

In [11]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

In [12]:
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    # device_map={"": torch.cuda.current_device()}, 
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)

config.json:   0%|          | 0.00/602 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/26.0M [00:00<?, ?B/s]

In [13]:
lora_config = LoraConfig(
    r=lora_r,
    lora_alpha=16,
    target_modules=lora_target_modules,  # Llama-style
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

In [14]:
from trl import SFTConfig, SFTTrainer

# Define ALL configuration parameters within SFTConfig
config = SFTConfig(
    # --- Standard TrainingArguments parameters ---
    output_dir=OUTPUT_DIR,
    learning_rate=lr,
    num_train_epochs=epochs,
    per_device_train_batch_size=bs,
    gradient_accumulation_steps=gradient_accumulation_steps,
    logging_steps=10,
    save_strategy="epoch",
    bf16=True,
    report_to="none",
    optim=optim,

    # --- SFT-specific parameters (moved here from previous iterations) ---
    dataset_text_field="text",
    max_length=max_length,
    packing=False,
)

# Initialize the SFTTrainer, passing the SFTConfig object to the 'args' parameter
trainer = SFTTrainer(
    model=model,
    args=config,  # Pass the combined SFTConfig object here
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=lora_config,
    # No other config parameters needed here
)

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

Adding EOS to train dataset:   0%|          | 0/4000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/4000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/4000 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

In [15]:
trainer.train()

ValueError: You can't train a model that has been loaded in 8-bit or 4-bit precision on a different device than the one you're training on. Make sure you loaded the model on the correct device using for example `device_map={'':torch.cuda.current_device()}` or `device_map={'':torch.xpu.current_device()}`

In [ ]:
ADAPTER_DIR = "./qlora-peft-output/adapter"

# 6. Save adapter weights

In [ ]:
# -----------------------------

# -----------------------------
os.makedirs(ADAPTER_DIR, exist_ok=True)
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print(f"Adapter saved to: {ADAPTER_DIR}")

In [ ]:
!tree


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel, AutoPeftModelForCausalLM

# -------------------------------------------------------
# Paths
# -------------------------------------------------------
BASE_ID = "meta-llama/Llama-3.1-8B-Instruct"
BASE_ID = "arnir0/Tiny-LLM"
BASE_ID = "Qwen/Qwen2-1.5B-Instruct"
ADAPTER_DIR = "./qlora-peft-output/adapter"
MERGED_DIR = "./merged-model"

# -------------------------------------------------------
# Load tokenizer
# -------------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained(
    BASE_ID,
    trust_remote_code=True
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# -------------------------------------------------------
# Load base model in fp16/bf16
# (merged model will be full precision LLM)
# -------------------------------------------------------
print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_ID,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
)

# -------------------------------------------------------
# Attach adapter
# -------------------------------------------------------
print("Loading adapter onto base model...")
model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_DIR,
)

# -------------------------------------------------------
# Merge adapter → base model
# -------------------------------------------------------
print("Merging LoRA adapter into base model weights...")
merged_model = model.merge_and_unload()   # <-- key line

# -------------------------------------------------------
# Save merged full model (optional)
# -------------------------------------------------------
merged_model.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)

print(f"\nMerged model saved to: {MERGED_DIR}\n")

In [ ]:
!tree

In [ ]:
def generate(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(merged_model.device)
    with torch.no_grad():
        outputs = merged_model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


# -------------------------------------------------------
# Run merged-model inference
# -------------------------------------------------------
print("==================== MERGED MODEL OUTPUT ====================\n")

for p in test_dataset:
    wrapped = (
        "You are a personalized assistant that knows details about the user based "
        "on prior fine-tuning data.\n\n"
        f"Question: {p['instruction']}\nAnswer:"
    )

    print(f"PROMPT: {p}\n")
    output = generate(wrapped)
    print(output)
    print("-" * 120)